# Hybrid Mamba-Attention — Chinese ↔ Vietnamese (Colab)

Notebook chạy **HybridMambaAttentionTranslator (~32.7M params)**:

* Encoder = **Bi-Mamba** (re-use `bi_mamba_mt.modules.bi_mamba.BiMambaBlock`).
* Decoder = **vanilla Transformer** (`nn.TransformerDecoder`, masked self-attn + cross-attn + FFN, pre-norm + GELU).
* Cùng tokenizer + cùng training loop với 2 baseline khác.

Mục đích chẩn đoán:

| Hybrid vs Bi-Mamba | Hybrid vs Transformer | Kết luận |
|---|---|---|
| ≈ Bi-Mamba | ≪ Transformer | bottleneck nằm ở encoder Bi-Mamba |
| ≫ Bi-Mamba | ≈ Transformer | bottleneck nằm ở decoder Mamba (cross-attn) |
| ≫ Bi-Mamba | ≫ Transformer (rare) | hybrid là kiến trúc tối ưu |

Notebook **không thay đổi gì trong `bi_mamba_mt/` hoặc `transformer_mt/`** — mọi thứ shared (data, tokenizer, evaluator, trainer) đến từ `mt_base/`.

## 1. Mount Google Drive (tuỳ chọn)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/bi_mamba_zh_vi'
    import os
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Mounted Drive at', DRIVE_DIR)
except Exception as e:
    print('Skip Drive (probably not running on Colab):', e)
    DRIVE_DIR = None

## 2. Clone repo

In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
WORK_DIR = '/content/NLP_DHM'

if not os.path.isdir(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
%cd {WORK_DIR}
!git pull --ff-only

## 3. Cài đặt dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

import sys
if 'src' not in sys.path:
    sys.path.insert(0, 'src')
print('sys.path[0] =', sys.path[0])


### 3b. (Tuỳ chọn) Cài CUDA fast-path cho Mamba

Hybrid encoder dùng `BiMambaBlock`, nên fast-path CUDA giúp giảm 30-50% thời gian train. Reference path PyTorch vẫn chạy được nếu skip cell này.

In [ ]:
# === Optional but recommended: CUDA fast-path (5–10× speedup) ===
# Repo code applies SiLU/gating outside the fused kernels so fast-path,
# reference forward, and step() keep the same semantics.
# `pip install mamba-ssm causal-conv1d` from PyPI tries to *build* from source
# on Colab and almost always fails (CUDA toolkit / torch ABI mismatch).
# Instead, install matching prebuilt wheels straight from the GitHub releases.
import sys, subprocess, torch

# Skip on CPU runtimes (no benefit, and the wheels need CUDA).
if not torch.cuda.is_available():
    print('No CUDA → skipping fast-path install (pure-PyTorch fallback will run).')
else:
    torch_mm  = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # e.g. '2.10'
    cuda_maj  = torch.version.cuda.split('.')[0]                          # e.g. '12'
    py_tag    = f'cp{sys.version_info.major}{sys.version_info.minor}'     # e.g. 'cp312'
    abi       = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
    suffix    = f'cu{cuda_maj}torch{torch_mm}cxx11abi{abi}-{py_tag}-{py_tag}-linux_x86_64.whl'

    ccv1d_ver = '1.6.1'
    mamba_ver = '2.3.1'
    ccv1d_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{ccv1d_ver}.post4/causal_conv1d-{ccv1d_ver}+{suffix}'
    mamba_url = f'https://github.com/state-spaces/mamba/releases/download/v{mamba_ver}/mamba_ssm-{mamba_ver}+{suffix}'

    print('Installing causal-conv1d:', ccv1d_url.rsplit("/", 1)[-1])
    r1 = subprocess.run(['pip', 'install', '-q', '--no-deps', ccv1d_url])
    print('Installing mamba-ssm:    ', mamba_url.rsplit("/", 1)[-1])
    r2 = subprocess.run(['pip', 'install', '-q', '--no-deps', mamba_url])

    ok = True
    try:
        import causal_conv1d, mamba_ssm  # noqa
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn  # noqa
        from causal_conv1d import causal_conv1d_fn  # noqa
    except Exception as e:
        ok = False
        print('FAST-PATH INSTALL FAILED:', e)
        print(' → falling back to pure-PyTorch Mamba (training still works, ~5–10× slower).')
    if ok:
        print('CUDA fast-path active: selective_scan_fn + causal_conv1d_fn imported OK.')


## 4. Cấu hình Colab

Dùng `configs/hybrid_mamba_attention.yaml` (cùng `data:` và `tokenizer:` với Bi-Mamba và Transformer baseline).

In [ ]:
import os, yaml, shutil
os.makedirs('configs', exist_ok=True)
shutil.copy('configs/hybrid_mamba_attention.yaml', 'configs/_colab.yaml')

with open('configs/_colab.yaml') as f:
    cfg = yaml.safe_load(f)

# T4 không hỗ trợ bf16 native → dùng fp16
cfg['train']['amp_dtype'] = 'fp16'
cfg['train']['num_workers'] = 4
cfg['train']['output_dir'] = 'runs/hybrid_mamba_attention'

with open('configs/_colab.yaml', 'w') as f:
    yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Config OK:')
print('  preset =', cfg['data']['preset'])
print('  d_model =', cfg['model']['d_model'])
print('  layers =', cfg['model']['n_encoder_layers'], '+', cfg['model']['n_decoder_layers'])
print('  encoder_d_ff =', cfg['model']['encoder_d_ff'])
print('  decoder_d_ff =', cfg['model']['decoder_d_ff'])
print('  max_steps =', cfg['train']['max_steps'])
print('  early_stopping_patience =', cfg['train']['early_stopping_patience'])
print('  output_dir =', cfg['train']['output_dir'])

## 5. Tải + chuẩn bị data (preset `everyday`)

Nếu đã chạy notebook Bi-Mamba/Transformer trước thì skip — `data/processed/*.jsonl` và `data/tokenizer/spm.model` dùng chung. Cần thiết để so sánh apples-to-apples.

In [ ]:
import os
PROC = 'data/processed'
if not (os.path.exists(f'{PROC}/train.jsonl') and os.path.exists('data/tokenizer/spm.model')):
    !python scripts/prepare_data.py --config configs/_colab.yaml --preset everyday
    !python scripts/train_tokenizer.py --config configs/_colab.yaml
else:
    print('Data + tokenizer đã có sẵn → skip.')
    !ls -lh data/processed/ data/tokenizer/

## 6. Khởi tạo model + kiểm tra số tham số

Phải xấp xỉ Bi-Mamba (32.43M) và Transformer (30.8M) để so sánh fair.

In [ ]:
from hybrid_mt.model import HybridMambaAttentionTranslator, ModelConfig
from mt_base.utils import count_parameters, human_format
import yaml

with open('configs/_colab.yaml') as f:
    cfg = yaml.safe_load(f)

mcfg = ModelConfig(**cfg['model'])
m = HybridMambaAttentionTranslator(mcfg)
n = count_parameters(m)
print(f'Hybrid params: {human_format(n)} ({n:,})')
print(f'  vs Bi-Mamba 32.43M / Transformer 30.8M')

# Sanity: confirm Mamba init guard preserved A_log/D
import torch
mb = m.encoder_layers[0].bi_mamba.fwd
expected_A = -torch.exp(torch.log(torch.arange(1, mcfg.d_state+1, dtype=torch.float)))
A = -torch.exp(mb.A_log.float())[0]
print(f'  A_log preserved : {torch.allclose(A, expected_A, atol=1e-3)}')
print(f'  D == ones       : {torch.allclose(mb.D, torch.ones_like(mb.D))}')
print(f'  dt_bias preserved: {getattr(mb.dt_proj.bias, "_no_reinit", False)}')

## 7. Train

In [ ]:
# rm -rf runs/hybrid_mamba_attention  # uncomment để clean restart
!python scripts/train_hybrid.py --config configs/_colab.yaml

## 8. Average last-5 + EMA checkpoints

In [ ]:
!python scripts/avg_ckpts.py --ckpts-dir runs/hybrid_mamba_attention --n 5
!python scripts/avg_ckpts.py --ckpts-dir runs/hybrid_mamba_attention --n 5 --ema
!ls -lh runs/hybrid_mamba_attention/*.pt | head -20

## 9. Đánh giá SacreBLEU + chrF (cả hai chiều, length-buckets)

So 6 candidate checkpoint (`best`, `best_ema`, `latest`, `latest_ema`, `avg_last5`, `avg_last5_ema`) trên 5000 câu test với beam=4.

In [ ]:
import os
ckpts = []
for name in ['best.pt', 'best_ema.pt', 'latest.pt', 'latest_ema.pt', 'avg_last5.pt', 'avg_last5_ema.pt']:
    p = f'runs/hybrid_mamba_attention/{name}'
    if os.path.exists(p):
        ckpts.append(p)

print('Eval candidates:', ckpts)
for ckpt in ckpts:
    print(f'\n=== {ckpt} (beam=4, default LP) ===')
    !python scripts/evaluate_hybrid.py --config configs/_colab.yaml \
        --checkpoint {ckpt} --num-samples 5000 --beam-size 4 --length-buckets

## 9b. Grid-sweep decoding (beam × length_penalty → CSV)

Dùng `scripts/sweep_decode.py --model-kind hybrid` (mới được extend trong PR này — backward-compat với Bi-Mamba sweep cũ).

In [ ]:
import os, csv

# Pick best available checkpoint
sweep_ckpt = None
for cand in ['avg_last5_ema.pt', 'best_ema.pt', 'avg_last5.pt', 'best.pt']:
    p = f'runs/hybrid_mamba_attention/{cand}'
    if os.path.exists(p):
        sweep_ckpt = p
        break
assert sweep_ckpt is not None, 'Run training first (cell 7).'

print(f'Sweep target: {sweep_ckpt}')
csv_path = 'runs/hybrid_mamba_attention/sweep.csv'

!python scripts/sweep_decode.py \
    --model-kind hybrid \
    --config configs/_colab.yaml \
    --checkpoint {sweep_ckpt} \
    --num-samples 2000 \
    --beams 1 2 4 6 \
    --lp-zh2vi 0.8 0.9 1.0 1.1 1.2 \
    --lp-vi2zh 0.6 0.8 0.9 1.0 \
    --length-buckets \
    --out {csv_path}

# Print top-3 per direction from CSV (filtered to bucket=='all').
import collections
rows = list(csv.DictReader(open(csv_path)))
by_dir = collections.defaultdict(list)
for r in rows:
    if r.get('bucket', 'all') == 'all':
        by_dir[r['direction']].append(r)
for d, rs in by_dir.items():
    rs.sort(key=lambda r: float(r['bleu']), reverse=True)
    print(f'\nTop-3 {d}:')
    for r in rs[:3]:
        print(f"  beam={r['beam']} lp={r['length_penalty']} BLEU={r['bleu']} chrF={r['chrf']} n={r['n']}")

## 10. Demo dịch (sanity check)

In [ ]:
import os
import torch
import yaml
from hybrid_mt.model import HybridMambaAttentionTranslator, ModelConfig
from mt_base.tokenizer import Tokenizer
from mt_base.translate import translate_batch

with open('configs/_colab.yaml') as f:
    cfg = yaml.safe_load(f)

ckpt_path = None
for cand in ['avg_last5_ema.pt', 'best_ema.pt', 'avg_last5.pt', 'best.pt']:
    p = f'runs/hybrid_mamba_attention/{cand}'
    if os.path.exists(p):
        ckpt_path = p
        break
assert ckpt_path is not None, 'Run training first (cell 7).'

print(f'Loading {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
mcfg = ModelConfig(**ckpt.get('model_cfg', cfg['model']))
m = HybridMambaAttentionTranslator(mcfg).cuda().eval()
m.load_state_dict(ckpt['model'], strict=False)

tok = Tokenizer('data/tokenizer/spm.model')

zh_samples = ['你好，世界。', '今天天气真好。', '这本书我已经读完了。']
vi_samples = ['Xin chào thế giới.', 'Hôm nay trời thật đẹp.', 'Tôi đã đọc xong cuốn sách này.']

print('--- zh → vi ---')
for s, t in zip(zh_samples, translate_batch(m, tok, zh_samples, 'zh2vi', beam_size=4, length_penalty=1.0)):
    print(f'  {s!r} -> {t!r}')

print('--- vi → zh ---')
for s, t in zip(vi_samples, translate_batch(m, tok, vi_samples, 'vi2zh', beam_size=4, length_penalty=0.9)):
    print(f'  {s!r} -> {t!r}')

## 11. (Tuỳ chọn) Lưu checkpoint vào Drive

In [ ]:
import shutil, os
if DRIVE_DIR:
    target = os.path.join(DRIVE_DIR, 'runs/hybrid_mamba_attention')
    os.makedirs(target, exist_ok=True)
    for f in ['best.pt', 'best_ema.pt', 'avg_last5.pt', 'avg_last5_ema.pt', 'final.pt', 'config.yaml', 'sweep.csv']:
        s = f'runs/hybrid_mamba_attention/{f}'
        if os.path.exists(s):
            shutil.copy(s, target)
            print('Copied', f)